In [4]:
from cellpose import models, io
from bioio import BioImage
import napari
from skimage.transform import rescale
from skimage.io import imsave
import random
from glob import glob

# Testing parameters for cellpose 4

In [ ]:
test_img = r"E:\KristinG\20260625_zebrafish\fin\WT_ctrl_2_skin_3_dorsal_fin_Out.czi"
img = BioImage(test_img)
img_rd = img.get_image_data("ZYX", C=1)

In [ ]:
img_twoch = img.get_image_data("ZYXC")

In [ ]:
img_twoch.shape

In [ ]:
cp = io.logger_setup()
model = models.CellposeModel(gpu=True)

In [ ]:
rescaled_img = rescale(img_rd, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True)

In [ ]:
new_scale = img.physical_pixel_sizes[0]*0.5, img.physical_pixel_sizes[1]*0.2, img.physical_pixel_sizes[2]*0.2

In [ ]:
anisotropy = new_scale[0]/new_scale[1]

In [ ]:
rescaled_masks_stitch, rescaled_flows_stitch, _ = model.eval(rescaled_img, z_axis=0, batch_size=32, do_3D=False,normalize=True,stitch_threshold=0.7,anisotropy=anisotropy)

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(rescaled_img, name='rescaled image', scale=new_scale,channel_axis=None)
v.add_labels(rescaled_masks_stitch, name='rescaled masks stitch 0.7 norm anisotropy', opacity=0.5,scale=new_scale)

In [ ]:
props = regionprops_table(rescaled_masks_stitch, properties=['label'])

In [ ]:
#v2.add_image(rescaled_img, name='rescaled image', scale=(img.physical_pixel_sizes[0]*(200/30), img.physical_pixel_sizes[1]*(200/30), img.physical_pixel_sizes[2]*(200/30)))
random_subset = []
while len(random_subset) < 10:
    r = randint(1, len(props['label']))
    if r not in random_subset:
        random_subset.append(r)

for i in random_subset:
    l = props['label'][i]
    obj = rescaled_masks_stitch == l
    v.add_labels(obj, name=f'object {l}', scale=new_scale, opacity=0.5)

In [ ]:
for l in props['equivalent_diameter_area']:
    print(l)

In [ ]:
save = r"E:\KristinG\20260625_zebrafish\fin_predictions"
imsave(r"E:\KristinG\20260625_zebrafish\fin_rescaled\DKO_ctrl_1_skin_1_ventral_fin_Out_rescaled.tif", rescaled_img.astype('uint16'))
imsave(r"E:\KristinG\20260625_zebrafish\fin_predictions\DKO_ctrl_1_skin_1_ventral_fin_Out_rescaled_masks_stitch.tif", rescaled_masks_stitch.astype('uint16'))

# Run cellpose4 on random subset of imgs

In [2]:
img_files = sorted(glob(r"E:\KristinG\20260625_zebrafish\fin\*.czi"))
wt_files = [f for f in img_files if "WT" in f]
dko_files = [f for f in img_files if "DKO" in f]
rand_WT = random.sample(wt_files, 5)
rand_DKO = random.sample(dko_files, 5)

In [3]:
wt_imgs = [BioImage(f).get_image_data("ZYX", C=1) for f in rand_WT]
dko_imgs = [BioImage(f).get_image_data("ZYX", C=1) for f in rand_DKO]
rescaled_wt_imgs = [rescale(img, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True) for img in wt_imgs]
rescaled_dko_imgs = [rescale(img, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True) for img in dko_imgs]
del(wt_imgs, dko_imgs)

In [ ]:
io.logger_setup()


In [7]:
model = models.CellposeModel(gpu=True)

2026-06-26 18:50:49,494 [core INFO] ** TORCH CUDA version installed and working. **
2026-06-26 18:50:49,496 [core INFO] >>>> using GPU (CUDA)
2026-06-26 18:50:51,614 [models INFO] >>>> loading model C:\Users\gkristin\.cellpose\models\cpsam_v2


In [8]:
all_imgs = rescaled_wt_imgs + rescaled_dko_imgs

In [ ]:
masks, _, _ = model.eval(all_imgs, batch_size=2, z_axis=0, do_3D=False,normalize=True,stitch_threshold=0.7)

2026-06-26 18:54:53,308 [utils INFO] 0%|          | 0/10 [00:00<?, ?it/s]


2026-06-26 18:54:53,328 [utils INFO] 
2026-06-26 18:54:53,330 [utils INFO] 0%|          | 0/8 [00:00<?, ?it/s]
2026-06-26 18:54:53,331 [utils INFO] 
2026-06-26 18:54:55,539 [utils INFO] 100%|##########| 8/8 [00:02<00:00,  3.63it/s]
2026-06-26 18:54:55,540 [models INFO] network run in 2.21s
2026-06-26 18:54:55,542 [utils INFO] 
2026-06-26 18:54:55,542 [utils INFO] 0%|          | 0/8 [00:00<?, ?it/s]
2026-06-26 18:54:55,543 [utils INFO] 


c:\Users\gkristin\AppData\Local\miniconda3\envs\cellpose4\Lib\site-packages\cellpose\dynamics.py:541: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


2026-06-26 18:54:56,291 [utils INFO] 100%|##########| 8/8 [00:00<00:00, 10.69it/s]
2026-06-26 18:54:56,292 [models INFO] stitching 8 planes using stitch_threshold=0.700 to make 3D masks


100%|██████████| 7/7 [00:00<00:00, 538.33it/s]

...

2026-06-26 18:55:23,502 [models INFO] masks created in 1.09s
2026-06-26 18:55:23,577 [utils INFO] 90%|######### | 9/10 [00:30<00:03,  3.36s/it]
2026-06-26 18:55:23,591 [utils INFO] 
2026-06-26 18:55:23,592 [utils INFO] 0%|          | 0/16 [00:00<?, ?it/s]
2026-06-26 18:55:23,593 [utils INFO] 


2026-06-26 18:55:26,341 [utils INFO] 100%|##########| 16/16 [00:02<00:00,  5.83it/s]
2026-06-26 18:55:26,342 [models INFO] network run in 2.75s
2026-06-26 18:55:26,343 [utils INFO] 
2026-06-26 18:55:26,343 [utils INFO] 0%|          | 0/16 [00:00<?, ?it/s]
2026-06-26 18:55:26,344 [utils INFO] 
2026-06-26 18:55:27,279 [utils INFO] 100%|##########| 16/16 [00:00<00:00, 17.10it/s]
2026-06-26 18:55:27,280 [models INFO] stitching 16 planes using stitch_threshold=0.700 to make 3D masks


100%|██████████| 15/15 [00:00<00:00, 437.28it/s]

2026-06-26 18:55:27,364 [models INFO] masks created in 1.02s
2026-06-26 18:55:27,455 [utils INFO] 100%|##########| 10/10 [00:34<00:00,  3.41s/it]


ValueError: not enough values to unpack (expected 3, got 2)

In [12]:
for i, img in enumerate(zip(masks, all_imgs)):
    m, img = img
    save = r"E:\KristinG\20260625_zebrafish\fin_predictions"
    imsave(rf"E:\KristinG\20260625_zebrafish\fin_predictions\{i}_rescaled_masks_stitch.tif", m.astype('uint16'))
    imsave(rf"E:\KristinG\20260625_zebrafish\fin_rescaled\{i}_rescaled.tif", img.astype('uint16'))

c:\Users\gkristin\AppData\Local\miniconda3\envs\cellpose4\Lib\site-packages\skimage\_shared\utils.py:386: UserWarning: E:\KristinG\20260625_zebrafish\fin_predictions\0_rescaled_masks_stitch.tif is a low contrast image
  return func(*args, **kwargs)
c:\Users\gkristin\AppData\Local\miniconda3\envs\cellpose4\Lib\site-packages\skimage\_shared\utils.py:386: UserWarning: E:\KristinG\20260625_zebrafish\fin_rescaled\0_rescaled.tif is a low contrast image
  return func(*args, **kwargs)
c:\Users\gkristin\AppData\Local\miniconda3\envs\cellpose4\Lib\site-packages\skimage\_shared\utils.py:386: UserWarning: E:\KristinG\20260625_zebrafish\fin_predictions\1_rescaled_masks_stitch.tif is a low contrast image
  return func(*args, **kwargs)
c:\Users\gkristin\AppData\Local\miniconda3\envs\cellpose4\Lib\site-packages\skimage\_shared\utils.py:386: UserWarning: E:\KristinG\20260625_zebrafish\fin_rescaled\1_rescaled.tif is a low contrast image
  return func(*args, **kwargs)
c:\Users\gkristin\AppData\Local\mini